# Correlation of blood pressure burden on DCI

## Preprocessing

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from utils.utils import load_encrypted_xlsx
from bp.bp_burden.analysis_utils import count_events, define_events_multiple_thresholds, multiple_duration_thresholds, event_count_to_DCI_coefficient, event_product_to_DCI_coefficient


In [ ]:
registry_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
bd_df_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke.csv'
pdms_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv'
outcome_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'

In [ ]:
registry_df = load_encrypted_xlsx(registry_data_path)
outcome_df = load_encrypted_xlsx(outcome_path)
bp_df = pd.read_csv(bd_df_path, sep= ';', decimal='.')
registry_pdms_correspondance_df = pd.read_csv(pdms_path)

In [ ]:
# drop duplicates 
bp_df = bp_df.drop_duplicates(subset=['pNr', 
                                       'systole',
                                       'diastole',
                                       'mitteldruck',
                                       'timeBd'])

registry_df.drop_duplicates(inplace=True)
registry_df.dropna(subset=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], inplace=True)

In [ ]:
bp_df=bp_df.merge(registry_pdms_correspondance_df, how='left', on='pNr')

bp_df['Date_birth']=pd.to_datetime(bp_df['Date_birth'], format='%d.%m.%Y')
outcome_df['Date_birth']=pd.to_datetime(outcome_df['Date_birth'])

outcome_df["mRS_FU_1y"]=pd.to_numeric(outcome_df['mRS_FU_1y'], errors='coerce')

In [ ]:
for pnr in tqdm(bp_df["pNr"].unique()):
    sos_center_nr = bp_df[bp_df["pNr"] == pnr]["SOS-CENTER-YEAR-NO."].values[0]
    name = bp_df[bp_df["pNr"] == pnr]["JoinedName"].values[0]
    date_birth = bp_df[bp_df["pNr"] == pnr]["Date_birth"].values[0]
    mrs_values = outcome_df[(outcome_df["SOS-CENTER-YEAR-NO."] == sos_center_nr) &
                        (outcome_df["Name"] == name) &
                        (outcome_df["Date_birth"] == date_birth)]["mRS_FU_1y"]
    if len(mrs_values) == 0:
        mrs = np.nan
    else:
        mrs = mrs_values.values[0]

    bp_df.loc[bp_df["pNr"] == pnr, "mrs_1y"] = mrs

In [ ]:
main_df = bp_df.merge(registry_df, 
                       left_on=['SOS-CENTER-YEAR-NO.', 'JoinedName', 'Date_birth'], 
                       right_on=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], 
                       how='left')

#### Compute timings

TODO: compute timings with T0 being ictus?

In [ ]:
main_df['Date_DCI_ischemia_first_image'] = pd.to_datetime(main_df['Date_DCI_ischemia_first_image'], errors='coerce', format='%Y-%m-%d')
main_df['Time_DCI_ischemia_first_image'] = pd.to_datetime(main_df['Time_DCI_ischemia_first_image'], errors='coerce', format='%H:%M:%S')

main_df['Date_DCI_infarct_first_image'] = pd.to_datetime(main_df['Date_DCI_infarct_first_image'], errors='coerce', format='%Y-%m-%d')
main_df['Date_DCI_infarct_first_image'] = pd. to_datetime(main_df['Date_DCI_infarct_first_image'],errors='coerce', format='%H:%M:%S')

main_df['timestamp_ischemia'] = pd.to_datetime(
    main_df['Date_DCI_ischemia_first_image'].astype(str) + ' ' + main_df['Time_DCI_ischemia_first_image'].astype(str),
    errors='coerce'
)

main_df['timestamp_infarction'] =  pd.to_datetime(
    main_df['Date_DCI_infarct_first_image'].astype(str) + ' ' + main_df['Time_DCI_infarct_first_image'].astype(str),
    errors='coerce'
)

main_df['timeBd']=pd.to_datetime(main_df['timeBd'], format='%Y-%m-%d %H:%M:%S.%f')

main_df['timeBd'] = main_df['timeBd'].dt.tz_localize(None)
main_df['timestamp_ischemia'] = main_df['timestamp_ischemia'].dt.tz_localize(None)

main_df['time_difference_ischemia']=main_df['timestamp_ischemia'] - main_df['timeBd']
main_df['time_difference_ischemia']=main_df['time_difference_ischemia'].dt.total_seconds() / 60

main_df['time_difference_infarction']=main_df['timestamp_infarction']-main_df['timeBd']
main_df['time_difference_infarction']=main_df['time_difference_infarction'].dt.total_seconds() / 60

main_df = main_df.sort_values(by=['pNr', 'timeBd'], ascending=True)

main_df['T0'] = main_df.groupby('pNr')['timeBd'].transform('min')
main_df['relative_time'] = main_df['timeBd'] - main_df['T0']
main_df['relative_time'] = main_df['relative_time'].dt.total_seconds() / 60
main_df['relative_time'] = pd.to_numeric(main_df['relative_time'], errors='coerce')

main_df['first_Th_relative_date'] = (pd.to_datetime(main_df['Date_First_Th']) - main_df['T0']).dt.total_seconds() / 60

#### Compute pressure x time product

In [ ]:
main_df['delta_time'] = main_df['timeBd'].shift(-1) - main_df['timeBd']
main_df['delta_time'] = main_df['delta_time'].dt.total_seconds() / 60

In [ ]:
main_df['product'] = main_df['delta_time'] * main_df['systole']


In [ ]:
working_df = main_df[['relative_time', 'pNr', 'delta_time', 'systole', 'product', 'DCI_YN_verified', 'mrs_1y', 'first_Th_relative_date']]

# Systolic blood pressure

#### First 24h

In [ ]:
# analysis for first 24 hours of monitoring
working_df_in_first_24h_monitoring = working_df[working_df['relative_time'] <= 24 * 60]  # 24 hours in minutes

events_df_in_first_24h_monitoring = define_events_multiple_thresholds(working_df_in_first_24h_monitoring,
                                           intensity_thresholds=[140, 150, 160, 170, 180, 190, 200, 210, 220],
                                           parameter_name='systole')

duration_thresholded_events_df_in_first_24h_monitoring = multiple_duration_thresholds(events_df_in_first_24h_monitoring,
                                                                                          duration_thresholds=[1, 5, 10, 15, 20, 30, 60, 120, 180])

event_counts_in_first_24h_monitoring_df = count_events(duration_thresholded_events_df_in_first_24h_monitoring)

event_count_association_with_DCI_first_24h_monitoring_df = event_count_to_DCI_coefficient(event_counts_in_first_24h_monitoring_df)

In [ ]:
# plot intensity threshold on x-axis, duration threshold on y-axis, and color as correlation coefficient (count to mRS_1y)

sns.heatmap(event_count_association_with_DCI_first_24h_monitoring_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='coefficient'
).reindex(index=sorted(event_count_association_with_DCI_first_24h_monitoring_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0)

In [ ]:
event_product_coefficient_DCI_first_24h_monitoring_df = event_product_to_DCI_coefficient(duration_thresholded_events_df_in_first_24h_monitoring, use_mixed_effects=True)

In [ ]:
sns.heatmap(event_product_coefficient_DCI_first_24h_monitoring_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='coefficient'
).reindex(index=sorted(event_product_coefficient_DCI_first_24h_monitoring_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0)

#### After 24h

In [ ]:
# analysis for 24h-to-end of monitoring

working_df_after_24h_monitoring = working_df[working_df['relative_time'] > 24 * 60]  # 24 hours in minutes
events_df_after_24h_monitoring = define_events_multiple_thresholds(working_df_after_24h_monitoring,
                                           intensity_thresholds=[140, 150, 160, 170, 180, 190, 200, 210, 220],
                                           parameter_name='systole')
duration_thresholded_events_df_after_24h_monitoring = multiple_duration_thresholds(events_df_after_24h_monitoring,
                                                                                          duration_thresholds=[1, 5, 10, 15, 20, 30, 60, 120, 180])
event_counts_after_24h_monitoring_df = count_events(duration_thresholded_events_df_after_24h_monitoring)

event_count_association_with_DCI_after_24h_monitoring_df = event_count_to_DCI_coefficient(event_counts_after_24h_monitoring_df)



In [ ]:
# plot intensity threshold on x-axis, duration threshold on y-axis, and color as correlation coefficient (count to mRS_1y)
sns.heatmap(event_count_association_with_DCI_after_24h_monitoring_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='coefficient'
).reindex(index=sorted(event_count_association_with_DCI_after_24h_monitoring_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0)

In [ ]:
event_product_coefficient_DCI_after_24h_monitoring_df = event_product_to_DCI_coefficient(duration_thresholded_events_df_after_24h_monitoring, use_mixed_effects=True)
sns.heatmap(event_product_coefficient_DCI_after_24h_monitoring_df.pivot_table(
    index='duration_threshold', 
    columns='intensity_threshold', 
    values='coefficient'
).reindex(index=sorted(event_product_coefficient_DCI_after_24h_monitoring_df['duration_threshold'].unique(), reverse=True)),
    annot=True, cmap='seismic', center=0)